<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS07_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS07 — Regresi untuk Prediksi Harga Properti

Jobsheet asli memakai contoh harga pizza dan dataset *Medical Cost Personal*. Versi PBL
ini memprediksi **harga properti** (target kontinu) dengan empat model: regresi linear
sederhana, regresi linear berganda, regresi polinomial, dan SVR. Sebagai pembanding
ditambahkan Random Forest.

Karena harga sangat miring, target dimodelkan dalam **log10(harga)**; hasil akhir
dikonversi kembali ke Rupiah agar mudah ditafsirkan.

## Langkah 1 — Muat Data dan Bagi Train/Test

Fitur: empat atribut fisik + `lokasi_freq` (proporsi kemunculan lokasi). Pembagian
80% latih / 20% uji memakai `random_state` tetap agar semua model memakai data uji
yang sama.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR_NUM] = df[FITUR_NUM].fillna(df[FITUR_NUM].median())
df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
df['lokasi_freq'] = df['lokasi_bersih'].map(df['lokasi_bersih'].value_counts(normalize=True))

FITUR = FITUR_NUM + ['lokasi_freq']
df['log_harga'] = np.log10(df['harga'])

X = df[FITUR]
y = df['log_harga']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape[0]:,} baris | Test: {X_test.shape[0]:,} baris')
print('Fitur:', FITUR)

Train: 44,936 baris | Test: 11,234 baris
Fitur: ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'lokasi_freq']


## Langkah 2 — Regresi Linear Sederhana (Satu Fitur)

Model `log10(harga) = b0 + b1 × luas_m2` dihitung manual dengan rumus OLS, lalu
dibandingkan dengan `LinearRegression` scikit-learn. Tujuannya memastikan rumus dan
library memberi hasil yang sama.

In [2]:
x = df['luas_m2']
y2 = df['log_harga']

b1 = np.cov(x, y2)[0, 1] / np.var(x)
b0 = y2.mean() - b1 * x.mean()

lr1 = LinearRegression().fit(df[['luas_m2']], y2)

print(f'Manual   : log_harga = {b0:.4f} + {b1:.6f} * luas_m2')
print(f'sklearn  : log_harga = {lr1.intercept_:.4f} + {lr1.coef_[0]:.6f} * luas_m2')
print(f'Artinya  : setiap tambahan 100 m2 menaikkan harga sekitar '
      f'{(10 ** (b1 * 100) - 1) * 100:.1f}%')

Manual   : log_harga = 8.7498 + 0.001067 * luas_m2
sklearn  : log_harga = 8.7498 + 0.001067 * luas_m2
Artinya  : setiap tambahan 100 m2 menaikkan harga sekitar 27.9%


In [3]:
sampel = df.sample(4000, random_state=RANDOM_STATE)
plt.figure(figsize=(8, 5))
plt.scatter(sampel['luas_m2'], sampel['log_harga'], s=10, alpha=0.35, label='data')
xs = np.linspace(df['luas_m2'].min(), df['luas_m2'].max(), 100)
plt.plot(xs, b0 + b1 * xs, color='red', label='garis regresi')
plt.xlabel('Luas tanah (m2)')
plt.ylabel('log10(harga)')
plt.title('Regresi linear sederhana: luas tanah vs log harga')
plt.legend()
plt.tight_layout()
plt.show()

## Langkah 3 — Fungsi Evaluasi

Semua model dievaluasi dengan metrik yang sama:

- **RMSE** dan **MAE** pada skala log (semakin kecil semakin baik).
- **R²** pada skala log.
- **MAE Rupiah** — rata-rata selisih harga prediksi vs aktual setelah dikonversi kembali.

MAE Rupiah mudah dibaca, tetapi sangat dipengaruhi rumah bernilai sangat besar; RMSE log
lebih stabil untuk membandingkan model.

In [4]:
def evaluasi_model(nama, model, X_tr, X_te, y_tr, y_te):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    waktu = time.perf_counter() - t0

    pred_log = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred_log))
    mae = mean_absolute_error(y_te, pred_log)
    r2 = r2_score(y_te, pred_log)

    harga_aktual = 10 ** y_te
    harga_pred = 10 ** pred_log
    mae_rp = mean_absolute_error(harga_aktual, harga_pred)

    return {'Model': nama, 'RMSE log': rmse, 'MAE log': mae, 'R2': r2,
            'MAE (Rp juta)': mae_rp / 1e6, 'Waktu fit (s)': waktu}


hasil = []
hasil.append(evaluasi_model('Linear (luas_m2)', LinearRegression(),
                             X_train[['luas_m2']], X_test[['luas_m2']], y_train, y_test))

model_berganda = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
hasil.append(evaluasi_model('Linear berganda (Ridge)', model_berganda,
                             X_train, X_test, y_train, y_test))

model_poli = make_pipeline(StandardScaler(),
                           PolynomialFeatures(degree=2, include_bias=False),
                           Ridge(alpha=1.0))
hasil.append(evaluasi_model('Polinomial derajat 2', model_poli,
                             X_train[['luas_m2', 'luas_bangunan']],
                             X_test[['luas_m2', 'luas_bangunan']], y_train, y_test))

pd.DataFrame(hasil).round(4)

,Model,RMSE log,MAE log,R2,MAE (Rp juta),Waktu fit (s)
0,Linear (luas_m2),0.3904,0.3166,0.2415,1272.3898,0.0026
1,Linear berganda (Ridge),0.2999,0.2288,0.5524,2925.9608,0.0057
2,Polinomial derajat 2,0.3113,0.2473,0.5177,967.5942,0.0062


## Langkah 4 — Support Vector Regression (SVR)

SVR dengan kernel RBF dilatih pada **sampel 10.000 baris** karena kompleksitasnya
naik cepat terhadap jumlah data. Data uji tetap memakai seluruh test set agar adil.

In [5]:
idx = np.random.default_rng(RANDOM_STATE).choice(len(X_train), 10_000, replace=False)
X_sub, y_sub = X_train.iloc[idx], y_train.iloc[idx]

svr = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale'))
hasil.append(evaluasi_model('SVR RBF (10k sampel)', svr, X_sub, X_test, y_sub, y_test))

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
hasil.append(evaluasi_model('Random Forest', rf, X_train, X_test, y_train, y_test))

tabel = pd.DataFrame(hasil).sort_values('RMSE log')
tabel.round(4)

,Model,RMSE log,MAE log,R2,MAE (Rp juta),Waktu fit (s)
4,Random Forest,0.1883,0.1287,0.8235,574.4767,2.6426
3,SVR RBF (10k sampel),0.2344,0.1718,0.7264,780.1776,3.3388
1,Linear berganda (Ridge),0.2999,0.2288,0.5524,2925.9608,0.0057
2,Polinomial derajat 2,0.3113,0.2473,0.5177,967.5942,0.0062
0,Linear (luas_m2),0.3904,0.3166,0.2415,1272.3898,0.0026


## Langkah 5 — Visualisasi Prediksi vs Aktual

Model terbaik (berdasarkan RMSE log) digambar pada sumbu prediksi vs aktual. Titik yang
ideal berada di garis diagonal. Sisaan (residual) diperiksa untuk melihat apakah error
masih berpola.

In [6]:
model_terbaik = rf
pred_log = model_terbaik.predict(X_test)
residual = y_test.values - pred_log

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].scatter(10 ** y_test, 10 ** pred_log, s=8, alpha=0.3)
batas = [50_000_000, 5e10]
ax[0].plot(batas, batas, 'r--', label='prediksi sempurna')
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('Harga aktual (Rp)')
ax[0].set_ylabel('Harga prediksi (Rp)')
ax[0].set_title('Random Forest: prediksi vs aktual')
ax[0].legend()

ax[1].hist(residual, bins=60, color='steelblue', edgecolor='white')
ax[1].axvline(0, color='red', linestyle='--')
ax[1].set_xlabel('Residual (log10)')
ax[1].set_title('Distribusi residual')

plt.tight_layout()
plt.show()

print(f'Rata-rata residual : {residual.mean():+.4f}')
print(f'Std residual       : {residual.std():.4f}')

Rata-rata residual : +0.0013
Std residual       : 0.1883


## Langkah 6 — Fitur Paling Berpengaruh

`feature_importances_` Random Forest menunjukkan seberapa sering sebuah fitur dipakai
untuk memisahkan data dan seberapa besar penurunan error yang dihasilkan.

In [7]:
imp = pd.Series(rf.feature_importances_, index=FITUR).sort_values()
plt.figure(figsize=(7, 3.5))
imp.plot(kind='barh', color='teal')
plt.xlabel('Kepentingan fitur')
plt.title('Feature importance — Random Forest')
plt.tight_layout()
plt.show()
print(imp.sort_values(ascending=False).round(4).to_string())

luas_bangunan         0.6564
lokasi_freq           0.1616
luas_m2               0.1085
jumlah_kamar_mandi    0.0499
jumlah_kamar          0.0236


## Langkah 7 — Tugas: Pengaruh Learning Rate pada Regresi Linear

Jobsheet asli meminta eksperimen learning rate. Untuk regresi linear, "learning rate"
muncul jika model dilatih dengan *gradient descent* (bukan closed-form). Berikut
perbandingan RMSE pada beberapa learning rate memakai `SGDRegressor`.

In [8]:
from sklearn.linear_model import SGDRegressor

baris = []
for lr in [0.0001, 0.001, 0.01, 0.1]:
    sgd = make_pipeline(StandardScaler(),
                        SGDRegressor(learning_rate='constant', eta0=lr,
                                     max_iter=2000, tol=1e-4, random_state=RANDOM_STATE))
    sgd.fit(X_train, y_train)
    pred = sgd.predict(X_test)
    baris.append({'learning rate': lr,
                  'RMSE log': np.sqrt(mean_squared_error(y_test, pred)),
                  'R2': r2_score(y_test, pred)})

pd.DataFrame(baris).round(4)

,learning rate,RMSE log,R2
0,0.0001,0.3001,5.518000e-01
1,0.0010,0.3020,5.461000e-01
2,0.0100,0.3071,5.305000e-01
3,0.1000,709.8487,-2.508081e+06


**Analisis JS07:**

- **Luas bangunan** dan **lokasi** adalah prediktor terkuat: feature importance Random Forest `luas_bangunan` 0,656, `lokasi_freq` 0,162, `luas_m2` 0,109, `jumlah_kamar_mandi` 0,050, `jumlah_kamar` 0,024.
- Regresi linear sederhana dengan `luas_m2` saja memberi R² 0,2415 (RMSE log 0,390); koefisien 0,00107 per m² berarti setiap tambahan 100 m² menaikkan harga ±27,9% (karena target log).
- **Regresi linear berganda (Ridge)** menaikkan R² ke **0,5524** (RMSE log 0,2999). **Polinomial derajat 2** sedikit lebih rendah (0,5177) — pada data ini, tambahan suku kuadrat/interaksi tidak membantu setelah regularisasi.
- **SVR RBF** (10.000 sampel) memberi R² **0,7264** (RMSE 0,2344) dan mengalahkan semua model linear, tetapi waktu latihnya 3,3 detik untuk 10 ribu baris — jauh lebih mahal daripada Ridge.
- **Random Forest terbaik**: R² **0,8235**, RMSE log **0,1883**, MAE log 0,1287, MAE Rupiah **Rp574 juta**. Hubungan harga dengan fitur properti tidak linear (pengaruh luas makin besar pada rumah mewah), sehingga model berbasis pohon lebih cocok.
- Catatan: MAE Rupiah Ridge (Rp2,9 miliar) jauh lebih besar daripada MAE log-nya karena kesalahan pada rumah bernilai puluhan miliar; metrik Rupiah sangat dipengaruhi outlier, sehingga RMSE log lebih adil untuk membandingkan model.
- **Learning rate SGD**: 0,0001/0,001/0,01 memberi R² 0,55/0,55/0,53 (stabil), sedangkan **0,1 divergen** (R² −2,5 juta). Learning rate terlalu besar membuat update melampaui minimum.
- Keterbatasan utama: **lokasi hanya diwakili frekuensi kemunculan**. Dua rumah identik di Sleman dan Jakarta Selatan bisa berbeda harga 5 kali. Model dengan one-hot lokasi (JS15) menutup sebagian celah ini.
